# Polarism on a Colab GPU

[Polarism](https://github.com/jakrog01/Polarism) simulates 2-D driven-dissipative Gross-Pitaevskii models of exciton-polariton condensates on CPU or CUDA GPUs.

This notebook runs two short GPU simulations: a condensate forming under a Gaussian pump, and a ring-shaped pump defined in a few lines of Python that traps vortices.

The first cell installs Polarism with the CuPy build matching the runtime's CUDA toolkit.

**Before running:** *Runtime → Change runtime type → T4 GPU* (any GPU works).

In [ ]:
import re, subprocess
cuda_major = re.search(r"release (\d+)", subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout).group(1)
%pip install -q "polarism[cuda{cuda_major}]"
!nvidia-smi -L

## 1. Condensate under a Gaussian pump

Lengths are in µm and times in ps. A continuous Gaussian pump above threshold feeds an incoherent reservoir; the condensate grows from small random noise. `ifrk-fft-cuda` is a spectral integrator running entirely on the GPU, and single precision is fast on consumer and Colab GPUs.

In [ ]:
import numpy as np
import cupy as cp
import matplotlib.pyplot as plt
import polarism as ps
from polarism.compute_engine import compute_engine


def make_config():
    cfg = ps.Config()
    cfg.grid.nx = cfg.grid.ny = 256
    cfg.grid.lx = cfg.grid.ly = 100.0
    cfg.grid.grid_type = "periodic"
    cfg.reservoir.reservoir_type = "single"
    cfg.laser.mode = "single"
    cfg.physics.init_mode = "complex_gaussian_zero_mean"
    cfg.physics.init_seed = 1
    cfg.solver.method = "ifrk-fft-cuda"
    cfg.solver.precision = "single"
    cfg.solver.dt = 0.05
    cfg.compute_engine.use_gpu = True
    return cfg


def show(sim):
    psi = cp.asnumpy(sim.state.psi)
    extent = [-50, 50, -50, 50]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))
    im = axes[0].imshow(np.abs(psi) ** 2, cmap="inferno", origin="lower", extent=extent)
    axes[0].set_title("density $|\\psi|^2$")
    fig.colorbar(im, ax=axes[0])
    im = axes[1].imshow(np.angle(psi), cmap="twilight", origin="lower", extent=extent)
    axes[1].set_title("phase arg $\\psi$")
    fig.colorbar(im, ax=axes[1])
    for ax in axes:
        ax.set_xlabel("x (µm)")
        ax.set_ylabel("y (µm)")
    plt.show()


cfg = make_config()
cfg.laser.laser_type = "continuous-gaussian"
cfg.laser.P0 = 0.1
cfg.laser.sigma_space = 15.0
cfg.solver.total_time = 1000.0
compute_engine.configure(cfg.compute_engine)

sim = ps.SimulationController(cfg)
sim.run()
show(sim)

The phase shows concentric rings: polaritons flow outward from the pumped spot.

## 2. Your own pump in ~20 lines

Every component (pump, reservoir, potential, solver, grid, absorption) lives in a registry. Subclass the base class, register it under a name, and select that name in the config. Here a ring-shaped pump traps the condensate inside the ring, where vortices form.

In [ ]:
from polarism.laser import register_laser
from polarism.laser.abstract_laser import AbstractLaser


@register_laser("ring")
class RingLaser(AbstractLaser):
    """Continuous Gaussian ring of radius `ring_radius` and width `sigma_space`."""

    def __init__(self, laser_config, X, Y, precision="double"):
        super().__init__(laser_config, X, Y, precision)
        self.radius = laser_config.extra.get("ring_radius", 20.0)
        self.width = laser_config.sigma_space
        self._finalize_spatial_envelope(X, Y)

    def _amplitude(self, t):
        return self.P0

    def _P_space(self, X, Y):
        r = self.xp.sqrt((X - self.x0) ** 2 + (Y - self.y0) ** 2)
        return self.xp.exp(-0.5 * ((r - self.radius) / self.width) ** 2)

    def _P_time(self, t):
        return 1.0

In [ ]:
cfg = make_config()
cfg.laser.laser_type = "ring"
cfg.laser.extra = {"ring_radius": 20.0}
cfg.laser.P0 = 0.1
cfg.laser.sigma_space = 4.0
cfg.solver.total_time = 1000.0

sim = ps.SimulationController(cfg)
sim.run()
show(sim)

Phase singularities (points where all colours meet) are quantized vortices. Change `init_seed`, `ring_radius` or `P0` and rerun to see different vortex configurations.

## Next steps

- Save HDF5/JSON/NPY output and animations: `cfg.result` ([output guide](https://jakrog01.github.io/Polarism/user-guide/output-and-visualization/)).
- Other solvers, reservoirs and pumps: [documentation](https://jakrog01.github.io/Polarism/).
- Write and share plugins: [plugin guide](https://jakrog01.github.io/Polarism/plugins/).
- Check `dt` and grid convergence before using results in a study.